# ABS3 → SLICES para MatText y MatGPT

Conversión de `data/results(1).json` mediante **SLICES de Xiao et al.**, sin reimplementar el algoritmo de grafos. Se usa la `structure` final guardada por cada cálculo, no la trayectoria. Se conservan identificadores, convergencia, energía y band gap, y se registran rechazos.

Se exportan dos serializaciones del mismo grafo: **MatGPT: estrategia 4** (etiquetas periódicas compactas, como `-o+`); **MatText: estrategia 3** (componentes separados, como `- o +`). Las dos se producen con la biblioteca. Se elimina el prefijo de grupo espacial añadido por SLICES reciente porque estos checkpoints no lo esperan.

El modo estricto comprueba que el JSON esté completo antes de convertir. Si una copia se interrumpe, se puede activar `ALLOW_PARTIAL_JSON=True` para recuperar únicamente los registros completos; la salida queda identificada como corpus parcial. Nunca se inventan coordenadas ni se reparan registros incompletos.

Fuentes: [SLICES oficial](https://github.com/xiaohang007/SLICES), [Xiao et al., Nature Communications (2023)](https://doi.org/10.1038/s41467-023-42870-7), [MatText: tokenizadores](https://github.com/lamalab-org/MatText/blob/2a594a4c69237da2daccab04e0515cf8cdcd715d/docs/tokenizers.md), [MatGPT](https://github.com/devalab/matgpt).

## 1. Entorno y configuración

Usar un kernel con `slices==2.0.12`, pymatgen, NumPy, pandas, tqdm y nbformat (versiones del proyecto en `requirements_tesis.txt`). SLICES importa TensorFlow y M3GNet incluso para codificar: sus dependencias deben ser compatibles. No se instala nada automáticamente.

`relax_model=None` evita inicializar el potencial de relajación: este notebook codifica y valida grafos, **no reconstruye ni relaja cristales**. Conviene ejecutarlo en un kernel separado del entrenamiento GPU, porque SLICES modifica variables del entorno al importarse.

In [1]:
from pathlib import Path
import hashlib
import json
import math
import sys
import platform
import warnings
from datetime import datetime, timezone
from importlib import metadata

# Localiza el proyecto desde la raíz o desde data/.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "data" / "results(1).json").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encuentra data/results(1).json desde este directorio.")
INPUT_PATH = PROJECT_ROOT / "data" / "results(1).json"
OUTPUT_ROOT = PROJECT_ROOT / "data" / "toSLICES_output"
ALLOW_PARTIAL_JSON = False  # True: solo objetos completos anteriores al fragmento roto.
MAX_RECORDS = None          # 3 para una prueba; None para todos los registros disponibles.
REQUIRE_CONVERGENCE = True
GRAPH_METHOD = "econnn"
# Mantener la celda original evita cambiar las etiquetas/estequiometría de las estructuras.
# No se reduce a celda primitiva ni se añade aumento de datos.
print(INPUT_PATH, "|", INPUT_PATH.stat().st_size, "bytes")


/users/0/cmadridl/projects/transformers-transfer-fine-tuning-thesis/data/results(1).json | 195161095 bytes


## 2. Lectura y auditoría del JSON

El esquema esperado es `{calculation_id: {results, structure, gap, ...}}`. La lectura usa `JSONDecoder` registro a registro para no materializar todas las trayectorias. Comprueba claves repetidas y el cierre del objeto. Si falta el cierre, el modo estricto no permite convertir; el modo parcial recupera exclusivamente objetos ya completos y registra dónde se detuvo.

In [2]:
def read_abs3(path, allow_partial=False):
    text = Path(path).read_text(encoding="utf-8-sig")
    decoder = json.JSONDecoder()
    def skip(i):
        while i < len(text) and text[i].isspace():
            i += 1
        return i
    pos = skip(0)
    if pos >= len(text) or text[pos] != "{":
        raise ValueError("Se esperaba un objeto JSON indexado por calculation_id.")
    pos += 1
    records, seen = [], set()
    failure = None
    after_comma = False
    while True:
        pos = skip(pos)
        if pos < len(text) and text[pos] == "}":
            if after_comma:
                failure = {"position": pos, "message": "Coma final no válida"}
                break
            pos = skip(pos + 1)
            if pos != len(text):
                failure = {"position": pos, "message": "Contenido después del objeto raíz"}
            break
        start = pos
        try:
            key, pos = decoder.raw_decode(text, pos)
            if not isinstance(key, str):
                raise ValueError("El identificador debe ser una cadena")
            if key in seen:
                raise ValueError(f"Identificador duplicado: {key}")
            pos = skip(pos)
            if pos >= len(text) or text[pos] != ":":
                raise ValueError("Falta ':' después del identificador")
            value, pos = decoder.raw_decode(text, skip(pos + 1))
        except (json.JSONDecodeError, ValueError) as exc:
            failure = {"position": getattr(exc, "pos", start), "message": str(exc)}
            break
        seen.add(key)
        # Omitir trajectory/forces/stress para limitar memoria; se mantiene structure final.
        record = ({k: value.get(k) for k in ("results", "structure", "gap")}
                  if isinstance(value, dict) else value)
        records.append((key, record))
        pos = skip(pos)
        after_comma = False
        if pos < len(text) and text[pos] == ",":
            pos += 1
            after_comma = True
        elif pos < len(text) and text[pos] == "}":
            continue
        else:
            failure = {"position": pos, "message": "Falta coma o cierre del objeto raíz"}
            break
    info = {"input": str(path), "input_bytes": Path(path).stat().st_size,
            "sha256": hashlib.sha256(Path(path).read_bytes()).hexdigest(),
            "complete_records": len(records), "partial_input": failure is not None,
            "parse_error": failure}
    if failure and not allow_partial:
        raise ValueError(f"JSON incompleto o inválido; {len(records)} registros completos. "
                         f"Detalle: {failure}. Vuelve a descargar el archivo o activa "
                         "ALLOW_PARTIAL_JSON=True para una recuperación PARCIAL explícita.")
    if failure:
        warnings.warn(f"CORPUS PARCIAL: {len(records)} registros completos. {failure}")
    if not records:
        raise ValueError("No hay registros completos para convertir")
    return records, info

records, input_audit = read_abs3(INPUT_PATH, ALLOW_PARTIAL_JSON)
print(json.dumps(input_audit, indent=2, ensure_ascii=False))
print("Primer identificador:", records[0][0])


{
  "input": "/users/0/cmadridl/projects/transformers-transfer-fine-tuning-thesis/data/results(1).json",
  "input_bytes": 195161095,
  "sha256": "c10c8c5378beaea03a33978da772329ff31e765b6c4a6da7d0c20bc588a92012",
  "complete_records": 480,
  "partial_input": false,
  "parse_error": null
}
Primer identificador: Ba1Ge1S3--Ba1Ge1S3_dist_perovskite--nm--gga-relax


## 3. Biblioteca oficial y vocabularios locales

Se registra la versión instalada y se comprueba su API. La exportación de MatGPT exige ausencia de tokens desconocidos y longitud compatible con cada checkpoint, contando inicio/fin y condiciones. No se cortan índices ni cadenas largas para forzar compatibilidad.

MatText no publica el tokenizador junto a los pesos: su CSV quedará **pendiente de validar los IDs y la longitud del tokenizador real**. La longitud por espacios es descriptiva, no sustituye a esa verificación. No se emplea el tokenizador de otro modelo.

In [3]:
import inspect
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from pymatgen.core import Structure
from slices.core import SLICES

versions = {name: metadata.version(name) for name in ["slices", "pymatgen", "numpy", "pandas"]}
print(versions)
if versions["slices"] != "2.0.12":
    warnings.warn("La API se ha revisado con SLICES 2.0.12; comprobar diferencias de versión.")
assert "relax_model" in inspect.signature(SLICES).parameters
converter = SLICES(graph_method=GRAPH_METHOD, check_results=False, relax_model=None)

checkpoint_specs = {
    "unconditional": ("unconditional", 0),
    "formation_bandgap": ("formation-bandgap", 2),
}
matgpt_models = {}
for name, (folder, nprops) in checkpoint_specs.items():
    paths = list((PROJECT_ROOT / "model-pretrained" / "matgpt" / "checkpoints" / folder).rglob("model_info*.json"))
    if len(paths) != 1:
        raise FileNotFoundError(f"Se esperaba un vocabulario de {name}; encontrados: {paths}")
    info = json.loads(paths[0].read_text())
    matgpt_models[name] = {"stoi": info["stoi"], "block_size": info["block_size"],
                           "nprops": nprops, "path": str(paths[0]),
                           "sha256": hashlib.sha256(paths[0].read_bytes()).hexdigest()}
print({name: {"context": m["block_size"], "vocabulary": len(m["stoi"])}
       for name, m in matgpt_models.items()})


Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead
{'slices': '2.0.12', 'pymatgen': '2025.10.7', 'numpy': '2.2.6', 'pandas': '2.3.3'}
{'unconditional': {'context': 442, 'vocabulary': 172}, 'formation_bandgap': {'context': 444, 'vocabulary': 172}}


## 4. Conversión, canonicalización y validación

Llamadas directas a la biblioteca: `Structure.from_dict`, `structure2SLICES`, `from_SLICES`, `to_SLICES`, `get_slices3`, `get_canonical_SLICES`, `check_SLICES_basic` y `check_SLICES`.

La canonicalización se usa para agrupar representaciones equivalentes, **sin eliminar filas**: un mismo grafo topológico puede corresponder a geometrías y etiquetas diferentes. Para exportar se exige el chequeo estricto 3D (incluyendo duplicados y rango del grafo), tanto para estrategia 4 como 3. Es validación del grafo por la biblioteca; no certifica reconstrucción geométrica, estabilidad ni coincidencia DFT.

`E_per_at` y `E_relax` se conservan como tales. `gap.bandgap` se exporta como `band_gap_eV`, sin asumir que sea indirecto. No se inventa `e_form` a partir de la energía total.

In [4]:
def finite_number(value):
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        return None
    return float(value) if math.isfinite(value) else None


def matgpt_compatibility(sequence, model):
    tokens = sequence.split()
    unknown = sorted(set(tokens) - set(model["stoi"]))
    total = len(tokens) + 2 + model["nprops"]  # <start>, <end>, condiciones
    return {"unknown": unknown, "length": total,
            "ok": not unknown and total <= model["block_size"]}


def convert_record(calculation_id, record):
    stage = "schema"
    row = {"calculation_id": calculation_id, "accepted": False, "status": "rejected"}
    try:
        if not isinstance(record, dict) or not isinstance(record.get("results"), dict):
            raise ValueError("Registro sin results válidos")
        results = record["results"]
        gap = record.get("gap") if isinstance(record.get("gap"), dict) else {}
        row.update(converged=results.get("convergence"),
                   source_formula=results.get("formula"),
                   E_per_at_eV=finite_number(results.get("E_per_at")),
                   E_relax_raw=finite_number(results.get("E_relax")),
                   band_gap_eV=finite_number(gap.get("bandgap")),
                   band_gap_is_direct=gap.get("is_direct"))
        if REQUIRE_CONVERGENCE and results.get("convergence") is not True:
            raise ValueError("Cálculo no convergido o sin convergencia confirmada")
        stage = "structure"
        structure = Structure.from_dict(record["structure"])
        if not structure.is_ordered or len(structure) == 0:
            raise ValueError("Estructura vacía o con ocupaciones desordenadas")
        if not np.isfinite(structure.lattice.matrix).all() or structure.volume <= 0:
            raise ValueError("Celda no válida")
        if not np.isfinite(structure.frac_coords).all():
            raise ValueError("Coordenadas no finitas")
        if not structure.is_valid(tol=0.5):
            raise ValueError("Distancia interatómica inferior a 0.5 Å")
        row.update(formula=structure.composition.reduced_formula, nsites=len(structure),
                   volume_A3=float(structure.volume))
        stage = "encode"
        raw = converter.structure2SLICES(structure, strategy=4)
        converter.from_SLICES(raw, strategy=4, fix_duplicate_edge=False)
        # El grafo no cambia: se omite únicamente el prefijo SLICES-PLUS de grupo espacial.
        converter.space_group_num = None
        legacy = converter.to_SLICES(strategy=4)
        canonical = converter.get_canonical_SLICES(legacy, strategy=4)
        s4 = " ".join(canonical.split())
        stage = "validate_strategy4"
        basic = bool(converter.check_SLICES_basic(s4, strategy=4))
        strict = bool(converter.check_SLICES(s4, strategy=4, dupli_check=True, graph_rank_check=True))
        row.update(slices_basic_valid=basic, slices_3d_valid=strict)
        if not basic or not strict:
            raise ValueError("La biblioteca rechaza la cadena SLICES estrategia 4")
        converter.from_SLICES(s4, strategy=4, fix_duplicate_edge=False)
        atoms4 = np.array(converter.atom_types, copy=True)
        edges4 = np.array(converter.edge_indices, copy=True)
        images4 = np.array(converter.to_jimages, copy=True)
        # En 2.0.12, el dispatcher to_SLICES(strategy=3) pasa un argumento extra.
        # Llamar al serializador oficial directamente evita ese fallo de la librería.
        s3 = " ".join(converter.get_slices3(
            converter.atom_symbols, converter.edge_indices, converter.to_jimages).split())
        stage = "validate_strategy3"
        valid3 = bool(converter.check_SLICES(s3, strategy=3, dupli_check=True, graph_rank_check=True))
        converter.from_SLICES(s3, strategy=3, fix_duplicate_edge=False)
        equivalent = (np.array_equal(atoms4, converter.atom_types)
                      and np.array_equal(edges4, converter.edge_indices)
                      and np.array_equal(images4, converter.to_jimages))
        if not valid3 or not equivalent:
            raise ValueError("La serialización de MatText no conserva el mismo grafo válido")
        if sorted(atoms4.tolist()) != sorted(structure.atomic_numbers):
            raise ValueError("La conversión no conserva la composición atómica")
        row.update(SLICES=s4, SLICES_mattext=s3,
                   topology_group=hashlib.sha256(s4.encode()).hexdigest(),
                   mattext_whitespace_length=len(s3.split()),
                   mattext_tokenizer_verified=False)
        for name, model in matgpt_models.items():
            c = matgpt_compatibility(s4, model)
            row.update({f"matgpt_{name}_compatible": c["ok"],
                        f"matgpt_{name}_length": c["length"],
                        f"matgpt_{name}_unknown_tokens": json.dumps(c["unknown"])})
        row.update(accepted=True, status="accepted", error_stage=None, error=None)
    except Exception as exc:
        row.update(error_stage=stage, error=f"{type(exc).__name__}: {exc}")
    return row


## 5. Convertir el corpus

Para una primera prueba, configurar `MAX_RECORDS=3`; para el corpus completo, `None`. Las excepciones de cada registro se anotan en la auditoría. No se interpretan como éxitos ni se sustituyen por cadenas vacías. Interrumpir el kernel cancela el proceso normalmente.

In [5]:
if MAX_RECORDS is not None and (not isinstance(MAX_RECORDS, int) or MAX_RECORDS < 1):
    raise ValueError("MAX_RECORDS debe ser None o un entero positivo")
selected_records = records if MAX_RECORDS is None else records[:MAX_RECORDS]
audit = pd.DataFrame([convert_record(key, record)
                      for key, record in tqdm(selected_records, desc="ABS3 → SLICES")])
valid = audit.loc[audit["accepted"].eq(True)].copy()
rejected = audit.loc[~audit["accepted"].eq(True)].copy()
print(f"Intentados: {len(audit)} | Válidos: {len(valid)} | Rechazados: {len(rejected)}")
display(audit.head())
if len(rejected):
    display(rejected[["calculation_id", "error_stage", "error"]])


ABS3 → SLICES:   0%|          | 0/480 [00:00<?, ?it/s]

ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!


ERROR - could not find a linearly independent cocycle basis!
ERROR - could not find a linearly independent cocycle basis!
ERROR - could not find a linearly independent cocycle basis!
ERROR - could not find a linearly independent cocycle basis!


ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!


ERROR - could not find a linearly independent cocycle basis!
ERROR - could not find a linearly independent cocycle basis!
ERROR - could not find a linearly independent cocycle basis!
ERROR - could not find a linearly independent cocycle basis!


ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!


... (21 outputs left out)

ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!
ERROR:root:Could not obtain the lattice basis from the cycle vectors!


ERROR - could not find a linearly independent cocycle basis!
ERROR - could not find a linearly independent cocycle basis!
ERROR - could not find a linearly independent cocycle basis!
ERROR - could not find a linearly independent cocycle basis!
Intentados: 480 | Válidos: 472 | Rechazados: 8


,calculation_id,accepted,status,converged,source_formula,E_per_at_eV,E_relax_raw,band_gap_eV,band_gap_is_direct,formula,...,mattext_whitespace_length,mattext_tokenizer_verified,matgpt_unconditional_compatible,matgpt_unconditional_length,matgpt_unconditional_unknown_tokens,matgpt_formation_bandgap_compatible,matgpt_formation_bandgap_length,matgpt_formation_bandgap_unknown_tokens,error_stage,error
0,Ba1Ge1S3--Ba1Ge1S3_dist_perovskite--nm--gga-relax,True,accepted,True,Ba1Ge1S3,-4.759775,-4.759775,0.5617,True,BaGeS3,...,350.0,False,True,220.0,[],True,222.0,[],None,None
1,Ba1Ge1S3--Ba1Ge1S3_dist_perovskite--nm--gga-st...,True,accepted,True,Ba1Ge1S3,-4.761982,-4.759775,0.5634,True,BaGeS3,...,350.0,False,True,220.0,[],True,222.0,[],None,None
2,Ba1Ge1S3--Ba1Ge1S3_hexagonal--nm--gga-relax,True,accepted,True,Ba1Ge1S3,-4.815511,-4.815511,2.0916,False,BaGeS3,...,250.0,False,True,160.0,[],True,162.0,[],None,None
3,Ba1Ge1S3--Ba1Ge1S3_hexagonal--nm--gga-static,True,accepted,True,Ba1Ge1S3,-4.819379,-4.815511,2.0916,False,BaGeS3,...,250.0,False,True,160.0,[],True,162.0,[],None,None
4,Ba1Ge1S3--Ba1Ge1S3_needle--nm--gga-relax,True,accepted,True,Ba1Ge1S3,-4.690139,-4.690139,0.4274,False,BaGeS3,...,340.0,False,True,214.0,[],True,216.0,[],None,None


,calculation_id,error_stage,error
190,Cu1S3Si1--Cu1S3Si1_needle--fm--gga-relax,validate_strategy4,ValueError: La biblioteca rechaza la cadena SL...
191,Cu1S3Si1--Cu1S3Si1_needle--fm--gga-static,validate_strategy4,ValueError: La biblioteca rechaza la cadena SL...
244,Ge1S3Zn1--Ge1S3Zn1_needle--nm--gga-relax,validate_strategy4,ValueError: La biblioteca rechaza la cadena SL...
245,Ge1S3Zn1--Ge1S3Zn1_needle--nm--gga-static,validate_strategy4,ValueError: La biblioteca rechaza la cadena SL...
392,S3Si1Sn1--S3Si1Sn1_hexagonal--nm--gga-relax,validate_strategy4,ValueError: La biblioteca rechaza la cadena SL...
393,S3Si1Sn1--S3Si1Sn1_hexagonal--nm--gga-static,validate_strategy4,ValueError: La biblioteca rechaza la cadena SL...
406,S3Si1Zn1--S3Si1Zn1_needle--nm--gga-relax,validate_strategy4,ValueError: La biblioteca rechaza la cadena SL...
407,S3Si1Zn1--S3Si1Zn1_needle--nm--gga-static,validate_strategy4,ValueError: La biblioteca rechaza la cadena SL...


## 6. Exportar y registrar procedencia

Cada ejecución crea una carpeta nueva. `abs3_slices_valid.csv` conserva las dos representaciones y las etiquetas; `audit.csv`/`rejected.csv` registran todas las decisiones. `matgpt_*.csv` incluye solamente cadenas compatibles con el checkpoint correspondiente; el archivo de dos condiciones **no implica disponer de energía de formación**. Para regresión puede usarse una cabeza nueva con las etiquetas originales. Los scripts de generación condicionada originales requieren adaptar columnas y objetivos: no renombrar `E_per_at` a `e_form` ni `band_gap_eV` a band gap indirecto sin evidencia.

`mattext_slices.csv` y `.jsonl` conservan las cadenas candidatas en el campo `SLICES`, con `mattext_tokenizer_verified=False`. Hasta recuperar/verificar el tokenizador original no se consideran entradas tokenizadas listas para entrenar. No se truncan secuencias ni se generan particiones aleatorias; `topology_group` ayuda a evitar fugas de representaciones equivalentes, aunque no sustituye la búsqueda de duplicados geométricos.

In [6]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
limited = len(selected_records) < len(records)
suffix = "_partial" if input_audit["partial_input"] or limited else ""
RUN_DIR = OUTPUT_ROOT / (RUN_ID + suffix)
RUN_DIR.mkdir(parents=True, exist_ok=False)
audit.to_csv(RUN_DIR / "audit.csv", index=False)
rejected.to_csv(RUN_DIR / "rejected.csv", index=False)
valid.to_csv(RUN_DIR / "abs3_slices_valid.csv", index=False)
export_counts = {}
if len(valid):
    base_columns = ["calculation_id", "topology_group", "formula", "nsites", "converged",
                    "E_per_at_eV", "E_relax_raw", "band_gap_eV", "band_gap_is_direct"]
    mattext = valid[base_columns + ["SLICES_mattext", "mattext_tokenizer_verified"]].rename(
        columns={"SLICES_mattext": "SLICES"})
    mattext.to_csv(RUN_DIR / "mattext_slices.csv", index=False)
    mattext.to_json(RUN_DIR / "mattext_slices.jsonl", orient="records", lines=True, force_ascii=False)
    export_counts["mattext_pending_tokenizer"] = len(mattext)
    for name in matgpt_models:
        subset = valid.loc[valid[f"matgpt_{name}_compatible"].eq(True), base_columns + ["SLICES"]]
        subset.to_csv(RUN_DIR / f"matgpt_{name}.csv", index=False)
        export_counts[f"matgpt_{name}"] = len(subset)
summary = {
    "source": input_audit, "run_id": RUN_ID, "versions": versions,
    "python": sys.version, "platform": platform.platform(),
    "settings": {"graph_method": GRAPH_METHOD, "relax_model": None,
                 "require_convergence": REQUIRE_CONVERGENCE,
                 "max_records": MAX_RECORDS, "allow_partial_json": ALLOW_PARTIAL_JSON,
                 "primitive_reduction": False, "canonicalize": True},
    "attempted": len(audit), "accepted": len(valid), "rejected": len(rejected),
    "limited_run": limited, "complete_corpus": not input_audit["partial_input"] and not limited,
    "exports": export_counts, "mattext_tokenizer_verified": False,
    "checkpoints": {k: {a: b for a, b in v.items() if a != "stoi"} for k, v in matgpt_models.items()},
    "notes": ["E_per_at no es energía de formación", "No se reconstruyen estructuras",
              "No se certifica equivalencia geométrica ni calidad científica"]}
(RUN_DIR / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(RUN_DIR)
print(json.dumps(export_counts, indent=2))
if not len(valid):
    warnings.warn("Ningún registro superó los filtros; consultar rejected.csv. No hay corpus entrenable.")


/users/0/cmadridl/projects/transformers-transfer-fine-tuning-thesis/data/toSLICES_output/20261019T213302_981691Z
{
  "mattext_pending_tokenizer": 472,
  "matgpt_unconditional": 472,
  "matgpt_formation_bandgap": 472
}
